In [1]:
import json
# from model.speech_pipeline import SpeechPipeline
from indexing.chunking import VADVTimeGapChunker
import os
from indexing.embed import process_embedding_batches

In [2]:
with open("/Users/daniyalkhan/Documents/WORK-for-Compassion/projects/RAG-allama_audio/data/meta_data/youtube_processing_log.json", 'r') as f:
    downloaded_audio_data= json.load(f)

In [ ]:
speech_pipeline= SpeechPipeline(
    vad_repo_or_dir='snakers4/silero-vad', 
    vad_model_name='silero_vad', 
    asr_model_name= "ai4bharat/indicconformer_stt_hi_hybrid_rnnt_large"
)

meta_data= {}
for i in downloaded_audio_data:
    file_output_path= downloaded_audio_data[i]['final_output_path']
    if file_output_path and downloaded_audio_data[i]['status']== 'completed':
        transcribed_vad_segments= speech_pipeline.process_audio_file(
            filepath= file_output_path,
            decoder= "ctc",
            language_id= "hi",
            vad_threshold= 0.3,
            min_speech_duration_ms= 500,
        )

        output_file= file_output_path.replace('.mp3', '.json').replace("audios", "transcriptions")
        with open(output_file, 'w') as f:
            json.dump(transcribed_vad_segments, f, ensure_ascii= False, indent= 4)
            
        meta_data[i]= downloaded_audio_data[i]
        meta_data[i]['speech_pipeline_status']= 'completed'
        meta_data[i]['speech_pipeline_output']= transcribed_vad_segments
        meta_data[i]['speech_pipeline_output_path']= output_file

In [6]:
meta_data.keys()

dict_keys(['c58bd53dcaf027ebe8f2b84b466a46f5'])

In [7]:
meta_data['c58bd53dcaf027ebe8f2b84b466a46f5'].keys()

dict_keys(['url', 'url_id', 'status', 'steps_completed', 'created_at', 'last_updated', 'error', 'final_output_path', 'filename', 'speech_pipeline_status', 'speech_pipeline_output', 'speech_pipeline_output_path'])

In [3]:
data_dir= "/Users/daniyalkhan/Documents/WORK-for-Compassion/projects/RAG-allama_audio/data/processed_data"
file_dir= os.path.join(data_dir, 'transcriptions')
file_name= "Dars-e-Quran┇Surah Al-Ahzaab 56-58┇The real meaning of ‘Salat’ on the Prophet.json"
file_path= os.path.join(file_dir, file_name)

with open(file_path, 'r') as f:
    vad_segments= json.load(f) 


In [4]:
splitter= VADVTimeGapChunker(file_name= file_name, gap_threshold= 0.85, max_tokens= 500)
chunks = splitter.split_documents(vad_segments)

In [5]:
chunks[:2]

[Document(metadata={'start': 0.0, 'end': 146.9, 'token_count': 512, 'file_name': 'Dars-e-Quran┇Surah Al-Ahzaab 56-58┇The real meaning of ‘Salat’ on the Prophet'}, page_content=' अल्लाह मस अल्लेह अ्लाह मोहम्मद है अल्लाह आलए मोहम्मद वो इसी को तो कहते है और ठीक है आप सब आ करे इसमें कोई हर्ज नहीं है लेकिन इसको महसूस कर देना शियों के लिएब तो फिर हम वैसे नहीं गयेंगे जैसे य शिया करते हमारी तरहसी डिफरेंट होगी इनके आम तौरऐ ऐसे होती है तो य बिल्कुल वो है जो आप को सिखाई गयी है और आप सैकड़ों तरीके से कर सकते है और बहुत ईमान लाने वालों को सबको इनक्लूड कर सकते है इसमें बिलकुल कोई हर्ज नहीं है  सब्सक्राइब द चैनल एंड प्रेस द लव आई कन टगेट एनंड अपडेट  खज बिल्ला शैत रजीम बफ्लाफीम इल्लाह बलाय का तो ह सल्लूनल नबी या अ लदीन अमन स्लूाल ही वसलीमो तस्लीमा ये आय थे सूर्य अहजाब की आयत नंबर फिफ्टी सिक्स छप्पन बेशक अल्लाह और उसके फरिश्ते नवी आरोप सलाद भेजते है इईमान लाने वालो तुम भी उनपरो सलाद भेजो और सलाम भेजो बहुत अच्छे तरीके ऐसी ये लवी तर्जुमा है जो अरबी के अल्फाास का  जरा तफसीली मफूम में रुक कर लेता हूँ  इस स

In [6]:
from qdrant_client import QdrantClient
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_qdrant import QdrantVectorStore
from dotenv import load_dotenv

load_dotenv('/Users/daniyalkhan/Documents/WORK-for-Compassion/projects/RAG-allama_audio/.env')

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")
client = QdrantClient(path="/Users/daniyalkhan/Documents/WORK-for-Compassion/projects/RAG-allama_audio/data/langchain_qdrant")
# client.create_collection(
#     collection_name="allama_rag_dev",
#     vectors_config= models.VectorParams(size= 3072, distance= models.Distance.COSINE),
# )
vector_store = QdrantVectorStore(
    client=client,
    collection_name="allama_rag_dev",
    embedding=embeddings,
)

In [ ]:
# process_embedding_batches(chunks, vector_store.add_documents)

I0000 00:00:1759738565.035377   35774 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handlers
100%|██████████| 32/32 [00:00<00:00, 500812.42it/s]


Sending final batch of 32 docs with 18023 tokens.
[Document(metadata={'start': 0.0, 'end': 146.9, 'token_count': 512, 'file_name': 'Dars-e-Quran┇Surah Al-Ahzaab 56-58┇The real meaning of ‘Salat’ on the Prophet'}, page_content=' अल्लाह मस अल्लेह अ्लाह मोहम्मद है अल्लाह आलए मोहम्मद वो इसी को तो कहते है और ठीक है आप सब आ करे इसमें कोई हर्ज नहीं है लेकिन इसको महसूस कर देना शियों के लिएब तो फिर हम वैसे नहीं गयेंगे जैसे य शिया करते हमारी तरहसी डिफरेंट होगी इनके आम तौरऐ ऐसे होती है तो य बिल्कुल वो है जो आप को सिखाई गयी है और आप सैकड़ों तरीके से कर सकते है और बहुत ईमान लाने वालों को सबको इनक्लूड कर सकते है इसमें बिलकुल कोई हर्ज नहीं है  सब्सक्राइब द चैनल एंड प्रेस द लव आई कन टगेट एनंड अपडेट  खज बिल्ला शैत रजीम बफ्लाफीम इल्लाह बलाय का तो ह सल्लूनल नबी या अ लदीन अमन स्लूाल ही वसलीमो तस्लीमा ये आय थे सूर्य अहजाब की आयत नंबर फिफ्टी सिक्स छप्पन बेशक अल्लाह और उसके फरिश्ते नवी आरोप सलाद भेजते है इईमान लाने वालो तुम भी उनपरो सलाद भेजो और सलाम भेजो बहुत अच्छे तरीके ऐसी ये लवी तर्जुमा है जो अरबी के अल्

In [9]:
from rag_service.service import RAGService
from langchain_google_genai import ChatGoogleGenerativeAI

In [11]:
llm_model = ChatGoogleGenerativeAI(model='gemini-2.5-flash')

rag= RAGService(
    vector_store=vector_store,
    llm_model= llm_model
)

question= "what is salat or namaz that was given to prophet?"
res= rag.workflow(question)

In [12]:
print(res)

According to the provided transcript:

Salat was not merely "namaz" as commonly understood (the five daily prayers with specific rakats), but a "complete system" (mukammal nizam).

The literal meaning of "salat," according to the majority of linguists, is:
*   Dua (supplication)
*   Bestowing blessings
*   Honoring and showing respect
*   Helping someone grow, evolve, or thrive.

When it is said that Allah and His angels send "salat" upon the Prophet, it means they honor him, help him evolve, increase his respect, and pave his way.

In the context of the Prophet's responsibilities of prophethood, "qiyam" (standing) involved reflection and contemplation in the light of the Quran for a certain period, and "sajda" (prostration) meant bowing before Allah and living under His will. While "namaz" was also performed, it was emphasized that the Prophet was not made to pray all night, but to recite the Quran as much as was easy, to fulfill his responsibilities.

The text emphasizes that limitin